# AI-Driven Fintech & Digital Payments: Africa and USA
### Exploratory Data Analysis

AI-powered credit scoring, fraud detection and mobile-money platforms need three things before they can reach people:
**connectivity** (internet and mobile networks), **accounts** (people who can actually send and receive money digitally),
and **money flows** (remittances are the largest cross-border payment stream into many African economies).

This notebook explores five World Bank indicators for **15 African countries** (Nigeria, Kenya, Ghana, South Africa, Ethiopia,
Tanzania, Uganda, Senegal, Côte d'Ivoire, Cameroon, Egypt, Morocco, Tunisia, Rwanda, Zambia) and the **United States** as a benchmark:

| Indicator | Code | Meaning |
|---|---|---|
| Financial inclusion | `FX.OWN.TOTL.ZS` | Account ownership at a bank or mobile-money provider (% of adults 15+) |
| Internet users | `IT.NET.USER.ZS` | Individuals using the internet (% of population) |
| Mobile subscriptions | `IT.CEL.SETS.P2` | Mobile cellular subscriptions per 100 people |
| Remittances | `BX.TRF.PWKR.CD.DT` | Personal remittances received (current US$) |
| GDP per capita | `NY.GDP.PCAP.CD` | GDP per capita (current US$) |

Data is read from the SQLite database `data/fintech_africa_usa.db` built by the pipeline
(`ingest.py → transform.py → load.py`).

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

# Works whether the notebook is started from the project root or from notebooks/
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DB_PATH = PROJECT_ROOT / "data" / "fintech_africa_usa.db"
assert DB_PATH.exists(), f"Database not found at {DB_PATH} - run the pipeline first (ingest -> transform -> load)."

with sqlite3.connect(DB_PATH) as conn:
    master = pd.read_sql_query("SELECT * FROM fintech_master", conn)
    summary = pd.read_sql_query("SELECT * FROM country_summary", conn)
    raw = pd.read_sql_query("SELECT * FROM raw_indicators", conn)

# Short display names
summary["country"] = summary["country_name"].replace({"Egypt, Arab Rep.": "Egypt"})
master["country"] = master["country_name"].replace({"Egypt, Arab Rep.": "Egypt"})
REGION_COLORS = {"Africa": "#2a9d8f", "North America": "#e76f51"}
print(f"fintech_master: {master.shape}, country_summary: {summary.shape}, raw_indicators: {raw.shape}")

## 1. Data overview

The master table holds one row per country and year. Survey-based financial-inclusion data (Global Findex) only exists for
survey rounds (2014, 2017, 2021/2022, 2024), whereas connectivity and macro indicators are annual - hence the missing values
per column below.

In [ ]:
indicator_cols = ["financial_inclusion_pct", "internet_users_pct", "mobile_subs_per_100",
                  "remittances_usd", "gdp_per_capita_usd", "population"]

coverage = (raw.groupby("indicator_key")
               .agg(countries=("country_code", "nunique"), first_year=("year", "min"),
                    last_year=("year", "max"), observations=("value", "count")))
display(coverage)
display(master[indicator_cols].describe().T.round(2))
print("Missing values per column (country-year grid):")
print(master[indicator_cols].isna().sum())

In [ ]:
latest_cols = ["country", "region", "financial_inclusion_pct", "internet_users_pct", "mobile_subs_per_100",
               "remittances_usd", "gdp_per_capita_usd", "remittance_to_gdp_ratio", "fintech_readiness_score",
               "readiness_rank"]
summary[latest_cols].sort_values("readiness_rank").round(2)

## 2. Financial inclusion by country (latest Findex year)

Account ownership counts both bank accounts and mobile-money accounts, so it is the clearest single measure of how many
adults can transact digitally.

In [ ]:
fi = summary.dropna(subset=["financial_inclusion_pct"]).sort_values("financial_inclusion_pct")
fig, ax = plt.subplots(figsize=(10, 6.5))
bars = ax.barh(fi["country"], fi["financial_inclusion_pct"], color=fi["region"].map(REGION_COLORS))
africa_avg = fi.loc[fi["region"] == "Africa", "financial_inclusion_pct"].mean()
ax.axvline(africa_avg, color="#264653", ls="--", lw=1.2, label=f"Africa average ({africa_avg:.1f}%)")
for bar, (_, row) in zip(bars, fi.iterrows()):
    ax.text(bar.get_width() + 0.8, bar.get_y() + bar.get_height() / 2,
            f"{row.financial_inclusion_pct:.1f}% ({int(row.financial_inclusion_pct_year)})", va="center", fontsize=8)
ax.set_xlim(0, 110)
ax.set_xlabel("Adults with a financial or mobile-money account (%)")
ax.set_title("Financial inclusion by country - latest Global Findex round")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

**What it shows.** Kenya (≈90%), Ghana and South Africa (≈81%) come closest to the U.S. benchmark (≈97%), while
the North African economies Tunisia, Egypt and Morocco sit below 45%, even though they are among the most connected countries in
the sample. In East Africa, mobile-money platforms such as M-Pesa, MTN MoMo and Airtel Money have brought adults into the formal
financial system who never opened a bank account. Rwanda's value comes from the 2017 round, the latest one available for it.

## 3. Internet user growth, 2020-2024 (top 5 African countries + USA)

In [ ]:
net = master[(master["year"].between(2020, 2024)) & master["internet_users_pct"].notna()]
top5 = (summary[summary["region"] == "Africa"].nlargest(5, "internet_users_pct")["country"].tolist())
subset = net[net["country"].isin(top5 + ["United States"])]

fig, ax = plt.subplots(figsize=(10, 5.5))
for country, grp in subset.groupby("country"):
    style = dict(lw=3, ls="--", color=REGION_COLORS["North America"]) if country == "United States" else dict(lw=2)
    ax.plot(grp["year"], grp["internet_users_pct"], marker="o", label=country, **style)
ax.set_xticks(range(2020, 2025))
ax.set_ylabel("Internet users (% of population)")
ax.set_title("Internet adoption trends 2020-2024: top 5 African countries vs USA")
ax.legend(ncol=3, loc="lower right")
plt.tight_layout()
plt.show()

growth = (net.sort_values("year").groupby("country")["internet_users_pct"]
             .agg(first="first", last="last").assign(change_pp=lambda d: d["last"] - d["first"]))
growth.sort_values("change_pp", ascending=False).round(2)

**What it shows.** North Africa (Morocco, Tunisia, Egypt) and South Africa lead African internet adoption, and Morocco
is closing in on the U.S. level. In the 2020-2024 window, Ghana, Kenya, Tanzania and Nigeria added the most internet users in
percentage-point terms, which widens the market for app-based and AI-driven financial services such as
alternative-data credit scoring and chatbot banking.

## 4. Mobile subscriptions vs financial inclusion (bubble = remittance inflows)

In [ ]:
bubble = summary.dropna(subset=["mobile_subs_per_100", "financial_inclusion_pct", "remittances_usd"]).copy()
bubble["remittances_bn"] = (bubble["remittances_usd"] / 1e9).round(2)
fig = px.scatter(
    bubble, x="mobile_subs_per_100", y="financial_inclusion_pct", size="remittances_bn", color="region",
    color_discrete_map=REGION_COLORS, text="country", size_max=60,
    hover_data={"internet_users_pct": ":.1f", "gdp_per_capita_usd": ":,.0f", "remittances_bn": ":.2f"},
    labels={"mobile_subs_per_100": "Mobile subscriptions per 100 people",
            "financial_inclusion_pct": "Account ownership (% adults)",
            "remittances_bn": "Remittances (US$ bn)"},
    title="Mobile penetration vs financial inclusion (bubble size = remittance inflows)",
)
fig.update_traces(textposition="top center", textfont_size=10)
fig.update_layout(height=600, template="plotly_white")
fig.show()

r_all = bubble[["mobile_subs_per_100", "financial_inclusion_pct"]].corr().iloc[0, 1]
print(f"Pearson r (mobile subscriptions vs account ownership, n={len(bubble)}): {r_all:.2f}")

**What it shows.** Having lots of SIM cards does not by itself mean high inclusion. Côte d'Ivoire and South Africa both exceed
175 subscriptions per 100 people, but their inclusion rates are 58% and 81%. Uganda and Zambia reach more than 70% inclusion
with fewer subscriptions. The correlation is weak, so what moves inclusion is a **working mobile-money ecosystem**
(interoperable wallets, agent networks, supportive regulation), not raw handset counts. Egypt and Nigeria have the largest
remittance bubbles, which makes them prime corridors for digital remittance products.

## 5. Correlation matrix of all indicators

In [ ]:
corr_cols = {
    "financial_inclusion_pct": "Fin. inclusion", "internet_users_pct": "Internet users",
    "mobile_subs_per_100": "Mobile subs", "remittances_usd": "Remittances (US$)",
    "gdp_per_capita_usd": "GDP per capita", "remittance_to_gdp_ratio": "Remit. % GDP",
    "fintech_readiness_score": "Readiness score",
}
corr = summary[list(corr_cols)].rename(columns=corr_cols).corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, square=True,
            linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Correlation of latest indicator values (15 African countries + USA)")
plt.tight_layout()
plt.show()

**What it shows.** Financial inclusion has almost no linear relationship with internet use across these 16 countries. Kenya
(90% inclusion, 35% internet) and Tunisia (38% inclusion, 77% internet) sit at opposite corners. Among the raw indicators, GDP per capita
is the one most associated with inclusion. Remittance dependence (% of GDP) is *negatively* related to inclusion, so the
economies that rely most on diaspora money (Egypt, Senegal, Nigeria) still have large unbanked populations that could gain from
cheaper, AI-assisted digital remittance channels. With only 16 data points these are descriptive patterns, not causal estimates.

## 6. Remittance inflows by country (US$ billions, latest year)

In [ ]:
rem = summary.dropna(subset=["remittances_usd"]).sort_values("remittances_usd", ascending=False).copy()
rem["remittances_bn"] = rem["remittances_usd"] / 1e9
fig, ax = plt.subplots(figsize=(11, 5.5))
sns.barplot(data=rem, x="country", y="remittances_bn", hue="region", palette=REGION_COLORS, dodge=False, ax=ax)
for i, (_, row) in enumerate(rem.iterrows()):
    label = f"{row.remittances_bn:.1f}"
    if pd.notna(row.remittance_to_gdp_ratio):
        label += f"\n({row.remittance_to_gdp_ratio:.1f}% GDP)"
    ax.text(i, row.remittances_bn + 0.5, label, ha="center", fontsize=7.5)
ax.set_ylabel("Personal remittances received (US$ bn)")
ax.set_xlabel("")
ax.set_title("Remittance inflows by country - latest available year")
ax.set_ylim(0, rem["remittances_bn"].max() * 1.2)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

africa_total = rem.loc[rem["region"] == "Africa", "remittances_bn"].sum()
print(f"Total remittance inflows to the 15 African countries (latest year each): US$ {africa_total:.1f} bn")
rem[["country", "remittances_usd_year", "remittances_bn", "remittance_to_gdp_ratio"]].round(2)

**What it shows.** Egypt (≈US$41.5bn) and Nigeria (≈US$22.8bn) are the largest recipients in the sample, followed by Morocco.
Senegal and Egypt depend on remittances the most relative to the size of their economies (above 10% of GDP). The U.S. figure
only counts remittances *received*, because the World Bank series tracks inflows. The United States is one of the main
*sending* countries for diaspora flows into Nigeria, Ghana, Kenya and Ethiopia, so the U.S.-Africa payment corridor is where
AI-based FX pricing, compliance automation and fraud detection can cut transfer costs.

## 7. Fintech readiness score

In [ ]:
ready = summary.dropna(subset=["fintech_readiness_score"]).sort_values("fintech_readiness_score")
fig = px.bar(ready, x="fintech_readiness_score", y="country", color="region", orientation="h",
             color_discrete_map=REGION_COLORS, text=ready["fintech_readiness_score"].round(1),
             labels={"fintech_readiness_score": "Fintech readiness score (0-100)", "country": ""},
             title="Fintech readiness = 0.4·inclusion + 0.3·internet + 0.3·mobile (capped at 100)")
fig.update_layout(height=550, template="plotly_white")
fig.show()

## 8. Key findings

1. **Mobile money, not connectivity, drives inclusion in sub-Saharan Africa.** Kenya, Ghana, Uganda and Zambia have higher
   account ownership than far more connected North African economies. Inclusion and internet use are essentially uncorrelated
   across the sample.
2. **Fast inclusion gains since 2017.** Senegal (+34 pp), Zambia, Cameroon, Nigeria and Ghana (+23-27 pp) improved the most
   between the 2017 and 2024 Findex rounds, compared with +4 pp in the already saturated U.S. market.
3. **Remittances are a major cross-border payment stream.** The 15 African countries received more than US$100bn in their latest
   reported year. Egypt, Senegal, Nigeria, Morocco and Tunisia each receive remittances worth more than 5% of GDP.
4. **A clear readiness gap to the USA.** Even the top African performers (South Africa, Ghana) score 11-13 points below the U.S.
   benchmark on the composite readiness score. Ethiopia, Uganda and Rwanda trail by about 40-52 points, mainly because of
   low internet penetration.
5. **The North African paradox.** Morocco, Tunisia and Egypt combine high internet use with low account ownership, which makes
   them a large opportunity for digital onboarding, e-KYC and AI-based credit scoring.

## 9. Conclusions

- **For fintech and AI builders:** the strongest near-term opportunities are (a) diaspora remittance corridors from the U.S. into
  Nigeria, Egypt, Ghana, Kenya and Ethiopia, and (b) digital onboarding in high-connectivity, low-inclusion markets in North Africa.
- **For policymakers:** interoperable mobile-money rails and agent networks have done more for inclusion than raw connectivity.
  Data-driven KYC and alternative-data credit scoring can extend that progress to the remaining unbanked adults.
- **Limitations:** Findex data is only available in survey years, some countries lack the newest round (for example Rwanda after
  2017), and with 16 countries the correlations describe patterns rather than prove causes.